# Reproducing and extending PAIR — Google Colab

**Recommended notebook for the GitHub repository.** The same experiments as the submitted student project: 30×3 on 100 behaviors; 10×3 versus 5×6 on 20 shared behaviors across three seeds.

**Run one section at a time — do not select `Runtime → Run all`**: real-model experiments can last longer than one Colab GPU session. Each completed behavior is checkpointed to **your Google Drive**, so rerunning a batch skips saved work.

The output success flag is Llama Guard's `unsafe` label, **not** a human-confirmed goal completion. Use only authorized test models. Do not publish raw prompts, outputs, or checkpoint JSONs.


## 1. Clone your GitHub repository and install dependencies

Publish the provided repository first, then replace `YOUR-USERNAME` below with your GitHub account. No ZIP upload is needed.


In [ ]:
from pathlib import Path
import subprocess

GITHUB_USERNAME = "YOUR-USERNAME"  # CHANGE THIS after publishing
REPO_NAME = "PAIR-Reproducibility"  # CHANGE if you named it differently

if GITHUB_USERNAME == "YOUR-USERNAME":
    raise ValueError("Set GITHUB_USERNAME to your GitHub username first")

repo_dir = Path("/content") / REPO_NAME
if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", str(repo_dir)], check=True)
%cd {repo_dir}
%pip -q install -r requirements.txt

assert Path('run_experiment_resumable.py').exists()
print("Code loaded:", repo_dir)


## 2. Mount Google Drive for persistent checkpoints

Make sure you mount the **same Drive that has your old checkpoints** when resuming; otherwise the code will create a fresh folder. Use a separate output folder for new model configurations or dataset versions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/PAIR_Project_Results')
EXT_FOLDER = BASE / 'local_extension'
REP_FOLDER = BASE / 'local_reproduction'
EXT_FOLDER.mkdir(parents=True, exist_ok=True)
REP_FOLDER.mkdir(parents=True, exist_ok=True)
for name, folder in [('Extension', EXT_FOLDER), ('Reproduction', REP_FOLDER)]:
    print(name, len(list((folder / 'checkpoints').glob('*.json'))), 'checkpoints', folder)


## 3. Check GPU

The study used a Colab **Tesla T4 GPU** for local inference through Ollama. PAIR's search algorithm itself only needs black-box model calls; the GPU accelerates the local LLMs.


In [ ]:
!nvidia-smi


## 4. Install/start Ollama and download the models

A Colab runtime reset deletes programs and downloaded weights stored in its temporary disk. These cells are repeatable. They do not touch your Drive checkpoints.


In [ ]:
import os
import time
import shutil
import subprocess
import requests

if shutil.which('ollama') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'zstd'], check=True)
    subprocess.run('curl -fsSL https://ollama.com/install.sh -o /tmp/ollama_install.sh && bash /tmp/ollama_install.sh', shell=True, check=True)

# Restart the server after a previous failure. Keep calls serialized.
subprocess.run(['pkill', '-f', '^ollama serve'], check=False)
time.sleep(2)
env = os.environ.copy()
env['OLLAMA_MAX_LOADED_MODELS'] = '2'
env['OLLAMA_NUM_PARALLEL'] = '1'
env['OLLAMA_KEEP_ALIVE'] = '5m'
log = open('/tmp/ollama.log', 'a')
server = subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=subprocess.STDOUT, env=env)
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        if r.status_code == 200:
            print('Ollama is ready')
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama not ready. Inspect /tmp/ollama.log')


In [ ]:
!ollama pull llama3.2:3b
!ollama pull llama-guard3:1b
!ollama list


## 5. Download the official JBB-Behaviors dataset

The original experiments used the official JBB CSV; this notebook does not bundle its potentially harmful objectives. The SHA-256 is printed for reproducibility.


In [ ]:
from pathlib import Path
import urllib.request
import hashlib

Path('data').mkdir(exist_ok=True)
data_file = Path('data/harmful-behaviors.csv')
if not data_file.exists():
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/JailbreakBench/JBB-Behaviors/resolve/main/data/harmful-behaviors.csv',
        data_file
    )
print('SHA-256:', hashlib.sha256(data_file.read_bytes()).hexdigest())
print('Expected hash from our experiment:', '4a8ec6832056b631eb092dccc60d37a61c3d441268268888b3d006288afeffa1')
print('Size:', data_file.stat().st_size, 'bytes')


## 6. Optional: offline unit tests and a two-behavior local pilot

The unit tests do not use a GPU. The optional pilot uses real local models but is too small for a research conclusion.


In [ ]:
!python -m unittest discover -s tests -v


In [ ]:
# Optional model pilot; remove the comment marker to run.
# !python -u run_experiment_resumable.py --mode ollama --allow-local-tests --task extension --seeds 0 --start 0 --limit 2 --data data/harmful-behaviors.csv --out /content/PAIR_PILOT


## 7. Extension — one resumable batch per click

Two setups: **breadth 10×3** and **depth 5×6**, with **30 maximum target queries per behavior**. Run the cell repeatedly; it chooses the next 5 missing behaviors for the first unfinished seed (0, then 1, then 2). The full experiment has **120 behavior-level runs**.

If the old checkpoints were created with another dataset/model version, use a **different Drive folder**. Never delete checkpoints to try to fix a runtime crash.


In [ ]:
from run_experiment_resumable import load_behaviors, choose_20, checkpoint_path
import subprocess

batch_size = 5
chosen = choose_20(load_behaviors(str(data_file)))
next_batch = None

for seed in [0, 1, 2]:
    for start in range(0, len(chosen), batch_size):
        chunk = chosen[start:start + batch_size]
        pending = [b for b in chunk if any(
            not checkpoint_path(EXT_FOLDER, setup, seed, b['id']).exists()
            for setup in ('breadth', 'depth'))]
        if pending:
            next_batch = (seed, start, len(chunk))
            break
    if next_batch is not None:
        break

if next_batch is None:
    print('EXTENSION COMPLETE: all 120 checkpoints are present')
else:
    seed, start, count = next_batch
    print(f'Running seed={seed}, behavior positions {start+1}-{start+count}/20')
    subprocess.run([
        'python', '-u', 'run_experiment_resumable.py',
        '--mode', 'ollama', '--allow-local-tests', '--task', 'extension',
        '--seeds', str(seed), '--start', str(start), '--limit', str(count),
        '--data', str(data_file), '--out', str(EXT_FOLDER)
    ], check=True)
    print('Saved checkpoints:', len(list((EXT_FOLDER / 'checkpoints').glob('*.json'))), '/120')


## 8. Analyze extension results

This uses only saved CSV logs after all 120 runs are complete. It produces summary tables and figures in the Drive folder.


In [ ]:
import pandas as pd
from IPython.display import display

ext_count = len(list((EXT_FOLDER / 'checkpoints').glob('*.json')))
print('Extension checkpoints:', ext_count, '/120')
if ext_count == 120:
    subprocess.run(['python', 'analyze_results.py', '--input', str(EXT_FOLDER)], check=True)
    summary = pd.read_csv(EXT_FOLDER / 'summary_by_seed.csv')
    display(summary)
else:
    print('Complete the remaining extension batches before analyzing')


## 9. Reproduction — one resumable batch per click

100 behaviors × 30 streams × 3 iterations × seed 0 = **100 behavior-level runs**. Use `repro_batch_size=2` to reduce the time lost to interruptions. Click the cell repeatedly until it reports complete.


In [ ]:
from run_experiment_resumable import checkpoint_path, load_behaviors
import subprocess

repro_batch_size = 2
all_behaviors = load_behaviors(str(data_file))
next_batch = None
for start in range(0, len(all_behaviors), repro_batch_size):
    chunk = all_behaviors[start:start + repro_batch_size]
    if any(not checkpoint_path(REP_FOLDER, 'reproduction', 0, b['id']).exists() for b in chunk):
        next_batch = (start, len(chunk))
        break

if next_batch is None:
    print('REPRODUCTION COMPLETE: all 100 checkpoints are present')
else:
    start, count = next_batch
    print(f'Running reproduction positions {start+1}-{start+count}/100')
    subprocess.run([
        'python', '-u', 'run_experiment_resumable.py',
        '--mode', 'ollama', '--allow-local-tests', '--task', 'reproduction',
        '--seeds', '0', '--start', str(start), '--limit', str(count),
        '--data', str(data_file), '--out', str(REP_FOLDER)
    ], check=True)
    print('Saved checkpoints:', len(list((REP_FOLDER / 'checkpoints').glob('*.json'))), '/100')


## 10. Analyze reproduction results and view charts


In [ ]:
repro_count = len(list((REP_FOLDER / 'checkpoints').glob('*.json')))
print('Reproduction checkpoints:', repro_count, '/100')
if repro_count == 100:
    subprocess.run(['python', 'analyze_results.py', '--input', str(REP_FOLDER)], check=True)
    display(pd.read_csv(REP_FOLDER / 'summary_by_seed.csv'))
else:
    print('Complete remaining reproduction batches before final analysis')


In [ ]:
from IPython.display import Image, display
for folder in [EXT_FOLDER, REP_FOLDER]:
    for image in sorted((folder / 'figures').glob('*.png')):
        print(image.name)
        display(Image(filename=str(image)))


## Reporting reminders

- Local models: Llama 3.2 3B for attacker and target, Llama Guard 3 1B as judge; **not the original paper's checkpoints**.
- Colab GPU: record `nvidia-smi` output (our runs used Tesla T4). The GPU is for local inference, not for implementing PAIR's search loop.
- The `success` metric is an automatic `unsafe` classification, which can have false positives. Human review was not completed in the supplied dataset.
- Round-robin streams are independent but not concurrent. Do not compare wall clock speed directly with paper results.
- Only use authorized model endpoints; keep raw outputs and `checkpoints` on private Drive storage. For interpretation and final charts, see [`RESULTS.md`](../RESULTS.md) in the repository.
